## Summary Statistics
#### Table 1: Average Inflation Expectations Across Models

In [ ]:
import os
import pandas as pd
import glob
import numpy as np
from tabulate import tabulate
import ast
import re
import json

# Base path
# Get the current notebook directory
notebook_dir = os.getcwd()

# Create base_path by going up one level from the notebook directory
base_path = os.path.dirname(notebook_dir)

# Define data_dir relative to base_path
data_dir = os.path.join(base_path, "Data", "Results")

# Get all CSV files in the Data directory
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

# Variables to analyze
list_columns = ['answer.Q1_S_Before_list', 'answer.Q2_L_Before_list']
numeric_columns = ['answer.Q2_L_After', 'answer.Q1_S_After']

# Model name mapping
model_name_mapping = {
    "DeepSeek-V3(temp=1)": "DeepSeek-V3",
    "Meta-Llama-3-70B-Instruct(temp=0.5)": "Llama3-70B",
    "claude-3-7-sonnet-20250219(temp=0.5)": "Sonnet-3.7",
    "gpt-4o(temp=0.5)": "GPT-4o(t=0.5)",
    "gpt-4o(temp=1)": "GPT-4o(t=1.0)",
    "gpt-4o(temp=1.5)": "GPT-4o(t=1.5)",
    "gpt-4o-mini(temp=1)": "GPT-4o-mini",
    "claude-3-5-haiku-20241022(temp=0.5)": "Haiku-3.5",
    "gpt-4.1(temp=1)": "GPT-4.1",
}

# Function to get simplified model name from filename
def get_model_name(filename):
    for key in model_name_mapping.keys():
        if key in filename:
            return model_name_mapping[key]
    return filename  # Return original if no match found

# Improved function to safely parse list strings from CSV with better error handling
def parse_list(list_str, model_name=None, row_id=None):
    """
    Parse a string representation of a list into a list of floats.
    Includes special handling for DeepSeek-V3 data with problematic observations.
    """
    if pd.isna(list_str):
        return None
    
    # Skip the problematic observation only for DeepSeek-V3
    if model_name == "DeepSeek-V3" and row_id == 70119139:
        print(f"Skipping DeepSeek problematic observation with ID 70119139")
        return None
    
    try:
        # Remove any leading/trailing whitespace
        if isinstance(list_str, str):
            list_str = list_str.strip()
            
            parsed_list = None
            
            # Try parsing as JSON first
            try:
                result = json.loads(list_str)
                if isinstance(result, list):
                    parsed_list = result
            except (json.JSONDecodeError, ValueError, TypeError):
                # Try ast.literal_eval
                try:
                    result = ast.literal_eval(list_str)
                    if isinstance(result, list):
                        # Handle nested lists by flattening
                        if result and isinstance(result[0], list):
                            result = result[0]
                        parsed_list = result
                except (SyntaxError, ValueError, TypeError):
                    # Try regex as last resort to extract numbers
                    numbers = re.findall(r'\d+(?:\.\d+)?', list_str)
                    if numbers and len(numbers) >= 10:
                        parsed_list = [float(num) for num in numbers[:10]]
            
            # Check if we have a valid list with 10 elements
            if parsed_list and len(parsed_list) == 10:
                # Convert all elements to float
                numeric_list = []
                
                # For DeepSeek-V3, check for extreme values
                MAX_ALLOWED_VALUE = 100.0  # Set a reasonable maximum value for probabilities
                has_extreme_value = False
                
                for value in parsed_list:
                    try:
                        num_value = float(value)
                        # Only check for extreme values in DeepSeek-V3 data
                        if model_name == "DeepSeek-V3" and num_value > MAX_ALLOWED_VALUE:
                            print(f"Found extreme value: {num_value} in DeepSeek observation")
                            has_extreme_value = True
                            break
                        numeric_list.append(num_value)
                    except (ValueError, TypeError):
                        # Skip lists with non-numeric elements
                        return None
                
                if model_name == "DeepSeek-V3" and has_extreme_value:
                    return None
                
                if len(numeric_list) == 10:
                    return numeric_list
        
        return None
    except Exception as e:
        print(f"Error parsing list: {e}")
        return None

# Function to calculate bin statistics with better error handling
def calculate_bin_statistics(df, column, model_name=None, by_treatment=False):
    """Calculate statistics for a column containing inflation expectation lists."""
    if column not in df.columns:
        print(f"Column {column} not found in dataframe")
        return None
    
    # Define bin descriptions
    bin_descriptions = [
        "Inflation of 12% or more",
        "Inflation between 8% and 12%",
        "Inflation between 4% and 8%",
        "Inflation between 2% and 4%",
        "Inflation between 0% and 2%",
        "Deflation between 0% and 2%",
        "Deflation between 2% and 4%",
        "Deflation between 4% and 8%",
        "Deflation between 8% and 12%",
        "Deflation of 12% or more"
    ]
    
    # Define bin midpoints
    bin_midpoints = {
        "Inflation of 12% or more": 14.0,           # Midpoint of 12+ (using 16 as upper bound)
        "Inflation between 8% and 12%": 10.0,       # Midpoint of 8-12
        "Inflation between 4% and 8%": 6.0,         # Midpoint of 4-8
        "Inflation between 2% and 4%": 3.0,         # Midpoint of 2-4
        "Inflation between 0% and 2%": 1.0,         # Midpoint of 0-2
        "Deflation between 0% and 2%": -1.0,        # Midpoint of 0-(-2)
        "Deflation between 2% and 4%": -3.0,        # Midpoint of (-2)-(-4)
        "Deflation between 4% and 8%": -6.0,        # Midpoint of (-4)-(-8)
        "Deflation between 8% and 12%": -10.0,      # Midpoint of (-8)-(-12)
        "Deflation of 12% or more": -14.0           # Midpoint of (-12)- (using -16 as lower bound)
    }
    
    if not by_treatment:
        # Analysis for all data combined
        bin_counts = {bin_name: 0 for bin_name in bin_descriptions}
        total_observations = 0
        weighted_sum = 0
        squared_diff_sum = 0
        valid_lists = 0
        
        for idx, row in df.iterrows():
            if pd.isna(row[column]):
                continue
                
            # Pass the row ID to the parser if available
            row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
            parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
            
            if parsed_list and len(parsed_list) == 10:
                valid_lists += 1
                for i, value in enumerate(parsed_list):
                    if i < len(bin_descriptions):  # Ensure we don't go out of bounds
                        bin_name = bin_descriptions[i]
                        try:
                            val = float(value)
                            bin_counts[bin_name] += val
                            weighted_sum += val * bin_midpoints[bin_name]
                            total_observations += val
                        except (ValueError, TypeError):
                            # Skip invalid values
                            continue
        
        # Calculate mean using midpoint formula if observations exist
        mean = weighted_sum / total_observations if total_observations > 0 else None
        
        # Calculate standard deviation
        if mean is not None:
            for idx, row in df.iterrows():
                if pd.isna(row[column]):
                    continue
                    
                # Pass the row ID to the parser if available
                row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
                
                if parsed_list and len(parsed_list) == 10:
                    for i, value in enumerate(parsed_list):
                        if i < len(bin_descriptions):  # Ensure we don't go out of bounds
                            bin_name = bin_descriptions[i]
                            try:
                                val = float(value)
                                diff = bin_midpoints[bin_name] - mean
                                squared_diff_sum += val * (diff ** 2)
                            except (ValueError, TypeError):
                                continue
            
            variance = squared_diff_sum / total_observations if total_observations > 0 else 0
            std_dev = np.sqrt(variance)
        else:
            std_dev = None
        
        result = {
            'mean': mean,
            'std_dev': std_dev,
            'valid_lists': valid_lists
        }
        
        return result
    
    else:
        # Analysis grouped by scenario.treatment
        if 'scenario.treatment' not in df.columns:
            print("Cannot analyze by treatment: scenario.treatment column not found")
            return None
        
        # Group by treatment and calculate statistics for each group
        treatments = df['scenario.treatment'].dropna().unique()
        results_by_treatment = {}
        
        for treatment in treatments:
            treatment_df = df[df['scenario.treatment'] == treatment]
            
            bin_counts = {bin_name: 0 for bin_name in bin_descriptions}
            total_observations = 0
            weighted_sum = 0
            squared_diff_sum = 0
            valid_lists = 0
            
            for idx, row in treatment_df.iterrows():
                if pd.isna(row[column]):
                    continue
                    
                # Pass the row ID to the parser if available
                row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
                
                if parsed_list and len(parsed_list) == 10:
                    valid_lists += 1
                    for i, value in enumerate(parsed_list):
                        if i < len(bin_descriptions):  # Ensure we don't go out of bounds
                            bin_name = bin_descriptions[i]
                            try:
                                val = float(value)
                                bin_counts[bin_name] += val
                                weighted_sum += val * bin_midpoints[bin_name]
                                total_observations += val
                            except (ValueError, TypeError):
                                # Skip invalid values
                                continue
            
            # Calculate mean using midpoint formula
            mean = weighted_sum / total_observations if total_observations > 0 else None
            
            # Calculate standard deviation
            if mean is not None:
                for idx, row in treatment_df.iterrows():
                    if pd.isna(row[column]):
                        continue
                        
                    # Pass the row ID to the parser if available
                    row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                    parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
                    
                    if parsed_list and len(parsed_list) == 10:
                        for i, value in enumerate(parsed_list):
                            if i < len(bin_descriptions):
                                bin_name = bin_descriptions[i]
                                try:
                                    val = float(value)
                                    diff = bin_midpoints[bin_name] - mean
                                    squared_diff_sum += val * (diff ** 2)
                                except (ValueError, TypeError):
                                    continue
                
                variance = squared_diff_sum / total_observations if total_observations > 0 else 0
                std_dev = np.sqrt(variance)
            else:
                std_dev = None
            
            results_by_treatment[treatment] = {
                'mean': mean,
                'std_dev': std_dev,
                'valid_lists': valid_lists
            }
        
        return results_by_treatment

# Dictionary to store results by variable and treatment
results_by_variable = {}
# Dictionary to store total results by file and variable
total_results = {}

# Process each CSV file and collect results
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    short_filename = os.path.splitext(filename)[0]  # Remove extension
    model_name = get_model_name(short_filename)
    
    # Load the CSV file
    df = pd.read_csv(csv_file)
    total_obs = df.shape[0]
    
    print(f"\nProcessing {filename} (n={total_obs})...")
    
    # Check for the problematic observation in DeepSeek-V3 file
    if model_name == "DeepSeek-V3" and 'agent.userid' in df.columns:
        problematic_ids = [70119139]  # Add any other problematic IDs here
        for pid in problematic_ids:
            matching_rows = df[df['agent.userid'] == pid]
            if not matching_rows.empty:
                print(f"Found problematic observation with ID {pid} in {filename}")
                # Print the treatment for this observation
                if 'scenario.treatment' in matching_rows.columns:
                    treatment = matching_rows['scenario.treatment'].iloc[0]
                    print(f"Treatment for problematic observation: {treatment}")
                # Print the problematic data
                for column in list_columns:
                    if column in matching_rows.columns:
                        print(f"Problematic data in {column}: {matching_rows[column].iloc[0]}")
    
    # Process density forecast variables
    for column in list_columns:
        if column not in df.columns:
            print(f"  Column {column} not found in this file")
            continue
            
        if column not in results_by_variable:
            results_by_variable[column] = {}
            total_results[column] = {}
            
        # Calculate overall statistics for this file/variable
        try:
            overall_stats = calculate_bin_statistics(df, column, model_name=model_name, by_treatment=False)
            if overall_stats:
                total_results[column][model_name] = overall_stats
        except Exception as e:
            print(f"  Error processing overall stats for {column}: {e}")
            
        # Calculate statistics by treatment
        try:
            stats_by_treatment = calculate_bin_statistics(df, column, model_name=model_name, by_treatment=True)
            
            if stats_by_treatment:
                for treatment, stats in stats_by_treatment.items():
                    if treatment not in results_by_variable[column]:
                        results_by_variable[column][treatment] = {}
                    
                    if stats['mean'] is not None and stats['std_dev'] is not None:
                        results_by_variable[column][treatment][model_name] = {
                            'mean': stats['mean'],
                            'std_dev': stats['std_dev'],
                            'valid_lists': stats['valid_lists']
                        }
                    else:
                        results_by_variable[column][treatment][model_name] = {
                            'mean': None,
                            'std_dev': None,
                            'valid_lists': 0
                        }
        except Exception as e:
            print(f"  Error processing {column}: {e}")
    
    # Process numeric variables
    for column in numeric_columns:
        if column not in df.columns:
            print(f"  Column {column} not found in this file")
            continue
            
        if column not in results_by_variable:
            results_by_variable[column] = {}
            total_results[column] = {}
        
        # Convert column to numeric
        df[column] = pd.to_numeric(df[column], errors='coerce')
        
        # Calculate overall statistics for this file/variable
        overall_mean = df[column].mean()
        overall_std = df[column].std()
        valid_count = df[column].count()
        
        total_results[column][model_name] = {
            'mean': overall_mean,
            'std_dev': overall_std,
            'count': valid_count
        }
        
        # Calculate statistics by treatment
        if 'scenario.treatment' in df.columns:
            treatment_stats = df.groupby('scenario.treatment')[column].agg(['mean', 'std', 'count'])
            
            for treatment, row in treatment_stats.iterrows():
                if treatment not in results_by_variable[column]:
                    results_by_variable[column][treatment] = {}
                
                if not pd.isna(row['mean']) and not pd.isna(row['std']):
                    results_by_variable[column][treatment][model_name] = {
                        'mean': row['mean'],
                        'std_dev': row['std'],
                        'count': row['count']
                    }
                else:
                    results_by_variable[column][treatment][model_name] = {
                        'mean': None,
                        'std_dev': None,
                        'count': 0
                    }

# Get all model names and treatments for creating the tables
all_model_names = []
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    short_filename = os.path.splitext(filename)[0]
    model_name = get_model_name(short_filename)
    all_model_names.append(model_name)
all_model_names = sorted(list(set(all_model_names)))

all_treatments = set()
for var_results in results_by_variable.values():
    all_treatments.update(var_results.keys())
all_treatments = sorted(all_treatments)

# Display results for each variable
for column in list_columns + numeric_columns:
    if column not in results_by_variable:
        print(f"\n\nNo data available for {column}")
        continue
    
    print(f"\n\n===== Summary Statistics for {column} =====")
    
    # Prepare table data
    table_data = []
    for treatment in all_treatments:
        if treatment not in results_by_variable[column]:
            row = [treatment] + ["N/A" for _ in all_model_names]
            table_data.append(row)
            continue
        
        row = [treatment]
        for model_name in all_model_names:
            if model_name in results_by_variable[column][treatment]:
                stats = results_by_variable[column][treatment][model_name]
                if stats['mean'] is not None and stats['std_dev'] is not None:
                    # Format as mean (sd)
                    cell = f"{stats['mean']:.2f} ({stats['std_dev']:.2f})"
                    # Add n count for density forecast variables
                    if column in list_columns:
                        cell += f" n={stats['valid_lists']}"
                    else:  # numeric columns
                        cell += f" n={int(stats['count'])}"
                else:
                    cell = "N/A"
                row.append(cell)
            else:
                row.append("N/A")
        
        table_data.append(row)
    
    # Add total row
    total_row = ["Total"]
    for model_name in all_model_names:
        if model_name in total_results[column]:
            stats = total_results[column][model_name]
            if stats['mean'] is not None and stats['std_dev'] is not None:
                # Format as mean (sd)
                cell = f"{stats['mean']:.2f} ({stats['std_dev']:.2f})"
                # Add n count
                if column in list_columns:
                    cell += f" n={stats['valid_lists']}"
                else:  # numeric columns
                    cell += f" n={int(stats['count'])}"
            else:
                cell = "N/A"
            total_row.append(cell)
        else:
            total_row.append("N/A")
    
    # Sort rows by treatment and add total row at the end
    table_data.sort(key=lambda x: str(x[0]))
    table_data.append(total_row)
    
    # Create headers
    headers = ["Treatment"] + all_model_names
    
    # Display table
    print(tabulate(table_data, headers=headers, tablefmt='grid'))

## Summary Statistics by Demographic Variables
#### Table 2: Demographic Heterogeneity in Inflation Expectations Across LLMs

In [ ]:
import os
import pandas as pd
import glob
import numpy as np
from tabulate import tabulate
import ast
import re
import json

# Variables to analyze
list_columns = ['answer.Q1_S_Before_list', 'answer.Q2_L_Before_list']
numeric_columns = ['answer.Q2_L_After', 'answer.Q1_S_After']

# Model name mapping
model_name_mapping = {
    "DeepSeek-V3(temp=1)": "DeepSeek-V3",
    "Meta-Llama-3-70B-Instruct(temp=0.5)": "Llama3-70B",
    "claude-3-7-sonnet-20250219(temp=0.5)": "Sonnet-3.7",
    "gpt-4o(temp=0.5)": "GPT-4o(t=0.5)",
    "gpt-4o(temp=1)": "GPT-4o(t=1.0)",
    "gpt-4o(temp=1.5)": "GPT-4o(t=1.5)",
    "gpt-4o-mini(temp=1)": "GPT-4o-mini",
    "claude-3-5-haiku-20241022(temp=0.5)": "Haiku-3.5",
    "gpt-4.1(temp=1)": "GPT-4.1",
}

# SELECT WHICH MODELS TO INCLUDE IN OUTPUT
# Set to None or empty list to include all models
selected_models = None  # Example - set to None to include all models

# Function to get simplified model name from filename
def get_model_name(filename):
    for key in model_name_mapping.keys():
        if key in filename:
            return model_name_mapping[key]
    return filename  # Return original if no match found

# Function to create age groups
def create_age_group(age):
    if pd.isna(age):
        return "Unknown"
    try:
        age = float(age)
        if age < 25:
            return "18-24"
        elif age < 35:
            return "25-34"
        elif age < 45:
            return "35-44"
        elif age < 55:
            return "45-54"
        elif age < 65:
            return "55-64"
        else:
            return "65+"
    except:
        return "Unknown"

# Improved function to safely parse list strings from CSV with better error handling
def parse_list(list_str, model_name=None, row_id=None):
    """
    Parse a string representation of a list into a list of floats.
    Includes special handling for DeepSeek-V3 data with problematic observations.
    """
    if pd.isna(list_str):
        return None
    
    # Skip the problematic observation only for DeepSeek-V3
    if model_name == "DeepSeek-V3" and row_id == 70119139:
        print(f"Skipping DeepSeek problematic observation with ID 70119139")
        return None
    
    try:
        # Remove any leading/trailing whitespace
        if isinstance(list_str, str):
            list_str = list_str.strip()
            
            parsed_list = None
            
            # Try parsing as JSON first
            try:
                result = json.loads(list_str)
                if isinstance(result, list):
                    parsed_list = result
            except (json.JSONDecodeError, ValueError, TypeError):
                # Try ast.literal_eval
                try:
                    result = ast.literal_eval(list_str)
                    if isinstance(result, list):
                        # Handle nested lists by flattening
                        if result and isinstance(result[0], list):
                            result = result[0]
                        parsed_list = result
                except (SyntaxError, ValueError, TypeError):
                    # Try regex as last resort to extract numbers
                    numbers = re.findall(r'\d+(?:\.\d+)?', list_str)
                    if numbers and len(numbers) >= 10:
                        parsed_list = [float(num) for num in numbers[:10]]
            
            # Check if we have a valid list with 10 elements
            if parsed_list and len(parsed_list) == 10:
                # Convert all elements to float
                numeric_list = []
                
                # For DeepSeek-V3, check for extreme values
                MAX_ALLOWED_VALUE = 100.0  # Set a reasonable maximum value for probabilities
                has_extreme_value = False
                
                for value in parsed_list:
                    try:
                        num_value = float(value)
                        # Only check for extreme values in DeepSeek-V3 data
                        if model_name == "DeepSeek-V3" and num_value > MAX_ALLOWED_VALUE:
                            print(f"Found extreme value: {num_value} in DeepSeek observation")
                            has_extreme_value = True
                            break
                        numeric_list.append(num_value)
                    except (ValueError, TypeError):
                        # Skip lists with non-numeric elements
                        return None
                
                if model_name == "DeepSeek-V3" and has_extreme_value:
                    return None
                
                if len(numeric_list) == 10:
                    return numeric_list
        
        return None
    except Exception as e:
        print(f"Error parsing list: {e}")
        return None

# Function to calculate bin statistics with better error handling
def calculate_bin_statistics(df, column, model_name=None):
    """Calculate statistics for a column containing inflation expectation lists."""
    if column not in df.columns:
        print(f"Column {column} not found in dataframe")
        return None
    
    # Define bin descriptions and their midpoints (for expectation calculation)
    bin_descriptions = [
        "Inflation of 12% or more",
        "Inflation between 8% and 12%",
        "Inflation between 4% and 8%",
        "Inflation between 2% and 4%",
        "Inflation between 0% and 2%",
        "Deflation between 0% and 2%",
        "Deflation between 2% and 4%",
        "Deflation between 4% and 8%",
        "Deflation between 8% and 12%",
        "Deflation of 12% or more"
    ]
    
    # Define bin midpoints
    bin_midpoints = {
        "Inflation of 12% or more": 14.0,
        "Inflation between 8% and 12%": 10.0,
        "Inflation between 4% and 8%": 6.0,
        "Inflation between 2% and 4%": 3.0,
        "Inflation between 0% and 2%": 1.0,
        "Deflation between 0% and 2%": -1.0,
        "Deflation between 2% and 4%": -3.0,
        "Deflation between 4% and 8%": -6.0,
        "Deflation between 8% and 12%": -10.0,
        "Deflation of 12% or more": -14.0
    }
    
    # Analysis for all data combined
    bin_counts = {bin_name: 0 for bin_name in bin_descriptions}
    total_observations = 0
    weighted_sum = 0
    squared_diff_sum = 0
    valid_lists = 0
    
    for idx, row in df.iterrows():
        if pd.isna(row[column]):
            continue
            
        # Pass the row ID to the parser if available
        row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
        parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
        
        if parsed_list and len(parsed_list) == 10:
            valid_lists += 1
            for i, value in enumerate(parsed_list):
                if i < len(bin_descriptions):  # Ensure we don't go out of bounds
                    bin_name = bin_descriptions[i]
                    try:
                        val = float(value)
                        bin_counts[bin_name] += val
                        weighted_sum += val * bin_midpoints[bin_name]
                        total_observations += val
                    except (ValueError, TypeError):
                        # Skip invalid values
                        continue
    
    # Calculate mean using midpoint formula if observations exist
    mean = weighted_sum / total_observations if total_observations > 0 else None
    
    # Calculate standard deviation
    if mean is not None:
        for idx, row in df.iterrows():
            if pd.isna(row[column]):
                continue
                
            # Pass the row ID to the parser if available
            row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
            parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
            
            if parsed_list and len(parsed_list) == 10:
                for i, value in enumerate(parsed_list):
                    if i < len(bin_descriptions):  # Ensure we don't go out of bounds
                        bin_name = bin_descriptions[i]
                        try:
                            val = float(value)
                            diff = bin_midpoints[bin_name] - mean
                            squared_diff_sum += val * (diff ** 2)
                        except (ValueError, TypeError):
                            continue
        
        variance = squared_diff_sum / total_observations if total_observations > 0 else 0
        std_dev = np.sqrt(variance)
    else:
        std_dev = None
    
    result = {
        'mean': mean,
        'std_dev': std_dev,
        'valid_lists': valid_lists,
        'bin_counts': bin_counts,
        'total_observations': total_observations
    }
    
    return result

# Function to calculate model statistics (without demographic breakdowns)
def calculate_model_statistics(csv_files, list_columns, numeric_columns, selected_models=None):
    """Calculate summary statistics for each model across all respondents"""
    # Store results by model and variable
    model_results = {}
    
    # Process each CSV file
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        short_filename = os.path.splitext(filename)[0]
        model_name = get_model_name(short_filename)
        
        # Skip if this model is not in the selected list
        if selected_models and model_name not in selected_models:
            continue
            
        # Load the CSV file
        print(f"Processing {filename} for model comparison...")
        df = pd.read_csv(csv_file)
        
        if model_name not in model_results:
            model_results[model_name] = {}
        
        # Process list columns (density forecasts)
        for column in list_columns:
            if column not in df.columns:
                continue
                
            # Calculate statistics
            stats = calculate_bin_statistics(df, column, model_name=model_name)
            
            if stats and stats['mean'] is not None:
                model_results[model_name][column] = {
                    'mean': stats['mean'],
                    'std_dev': stats['std_dev'],
                    'count': stats['valid_lists'],
                    'type': 'list'
                }
        
        # Process numeric columns (point estimates)
        for column in numeric_columns:
            if column not in df.columns:
                continue
                
            # Convert column to numeric
            df[column] = pd.to_numeric(df[column], errors='coerce')
            
            # Calculate statistics
            mean_val = df[column].mean()
            median_val = df[column].median()
            std_val = df[column].std()
            count = df[column].count()
            
            if not pd.isna(mean_val):
                model_results[model_name][column] = {
                    'mean': mean_val,
                    'median': median_val,
                    'std_dev': std_val,
                    'count': count,
                    'type': 'numeric'
                }
    
    return model_results

# Function to display model comparison tables
def display_model_comparison(model_results, list_columns, numeric_columns):
    """Display comparison tables for models"""
    print("\n\n===== MODEL COMPARISON =====")
    
    # For each variable
    all_variables = list_columns + numeric_columns
    for variable in all_variables:
        print(f"\n--- {variable} ---")
        
        # Prepare table data
        table_data = []
        headers = ["Model", "Mean", "Median", "Std Dev", "Count"]
        
        for model_name, variables in sorted(model_results.items()):
            if variable in variables:
                stats = variables[variable]
                
                if stats['type'] == 'numeric':
                    row = [
                        model_name,
                        f"{stats['mean']:.2f}",
                        f"{stats['median']:.2f}",
                        f"{stats['std_dev']:.2f}",
                        f"{int(stats['count'])}"
                    ]
                else:  # list type
                    row = [
                        model_name,
                        f"{stats['mean']:.2f}",
                        "N/A",  # No median for list variables
                        f"{stats['std_dev']:.2f}",
                        f"{stats['count']}"
                    ]
                
                table_data.append(row)
        
        # Display table
        print(tabulate(table_data, headers=headers, tablefmt='grid'))

# Function to calculate demographic summary statistics
def calculate_demographic_stats(csv_files, list_columns, numeric_columns, selected_models=None):
    # Store results by demographic variable, category, and model
    demographic_results = {}
    
    # Track all valid model names
    all_model_names = []
    
    # Process each CSV file
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        short_filename = os.path.splitext(filename)[0]
        model_name = get_model_name(short_filename)
        
        # Skip if this model is not in the selected list
        if selected_models and model_name not in selected_models:
            continue
            
        all_model_names.append(model_name)
        
        # Load the CSV file
        print(f"Processing {filename} for demographic analysis...")
        df = pd.read_csv(csv_file)
        
        # Define demographic columns with specific matching for age
        demographic_columns = {
            'age_group': 'agent.age' if 'agent.age' in df.columns else None,
            'education': next((col for col in df.columns if 'education' in col.lower()), None),
            'gender': next((col for col in df.columns if 'gender' in col.lower()), None),
            'marital': next((col for col in df.columns if 'marital' in col.lower()), None),
            'income': next((col for col in df.columns if 'income' in col.lower()), None)
        }
        
        # Create age groups if age column exists
        if demographic_columns['age_group']:
            # Create a new column without SettingWithCopyWarning
            df = df.assign(age_group=df[demographic_columns['age_group']].apply(create_age_group))
            demographic_columns['age_group'] = 'age_group'
        
        # Process each demographic variable
        for demo_key, demo_col in demographic_columns.items():
            if not demo_col or demo_col not in df.columns:
                continue
                
            if demo_key not in demographic_results:
                demographic_results[demo_key] = {}
            
            # Get unique categories for this demographic
            categories = df[demo_col].dropna().unique()
            
            # Process each category
            for category in categories:
                if pd.isna(category) or category == "":
                    continue
                    
                if category not in demographic_results[demo_key]:
                    demographic_results[demo_key][category] = {}
                
                if model_name not in demographic_results[demo_key][category]:
                    demographic_results[demo_key][category][model_name] = {}
                
                # Filter dataframe for this category
                category_df = df[df[demo_col] == category].copy()  # Use copy() to avoid SettingWithCopyWarning
                
                # Process list columns (density forecasts)
                for column in list_columns:
                    if column not in category_df.columns:
                        continue
                    
                    # Calculate statistics
                    stats = calculate_bin_statistics(category_df, column, model_name=model_name)
                    
                    if stats and stats['mean'] is not None:
                        demographic_results[demo_key][category][model_name][column] = {
                            'mean': stats['mean'],
                            'median': None,  # No direct median for probability distributions
                            'std_dev': stats['std_dev'],
                            'count': stats['valid_lists']
                        }
                
                # Process numeric columns (point estimates)
                for column in numeric_columns:
                    if column not in category_df.columns:
                        continue
                    
                    # Convert column to numeric - using proper pandas methods to avoid warning
                    category_df.loc[:, column] = pd.to_numeric(category_df[column], errors='coerce')
                    
                    # Calculate statistics
                    mean_val = category_df[column].mean()
                    median_val = category_df[column].median()
                    std_val = category_df[column].std()
                    count = category_df[column].count()
                    
                    if not pd.isna(mean_val):
                        demographic_results[demo_key][category][model_name][column] = {
                            'mean': mean_val,
                            'median': median_val,
                            'std_dev': std_val,
                            'count': count
                        }
    
    return demographic_results, sorted(list(set(all_model_names)))

# Function to display demographic results in tables
def display_demographic_tables(demographic_results, all_model_names, list_columns, numeric_columns):
    # For each demographic variable
    for demo_key, categories in demographic_results.items():
        print(f"\n\n===== Summary Statistics by {demo_key.upper()} =====")
        
        # For each analyzed variable
        for column in list_columns + numeric_columns:
            print(f"\n--- {column} ---")
            
            # Prepare table data for this variable
            table_data = []
            categories_sorted = sorted(categories.keys())
            
            for category in categories_sorted:
                row = [category]
                
                for model_name in all_model_names:
                    cell = "N/A"
                    
                    if model_name in categories[category] and column in categories[category][model_name]:
                        stats = categories[category][model_name][column]
                        
                        # Format mean and std_dev
                        mean_str = f"{stats['mean']:.2f}"
                        
                        if column in numeric_columns and stats['median'] is not None:
                            # For numeric columns, include median
                            median_str = f"{stats['median']:.2f}"
                            std_str = f"{stats['std_dev']:.2f}"
                            cell = f"μ={mean_str}, med={median_str}\nσ={std_str}, n={int(stats['count'])}"
                        else:
                            # For list columns, no median
                            std_str = f"{stats['std_dev']:.2f}"
                            cell = f"μ={mean_str}\nσ={std_str}, n={int(stats['count'])}"
                    
                    row.append(cell)
                
                table_data.append(row)
            
            # Create headers
            headers = [demo_key.capitalize()] + all_model_names
            
            # Display table
            print(tabulate(table_data, headers=headers, tablefmt='grid'))

# Run the analysis
if __name__ == "__main__":
    # Set pandas options to avoid warning display
    pd.options.mode.chained_assignment = None  # Suppress SettingWithCopyWarning
    
    print("\n\n===== MODEL COMPARISON ANALYSIS =====")
    
    # Calculate model statistics
    model_results = calculate_model_statistics(csv_files, list_columns, numeric_columns, selected_models)
    
    # Display model comparison
    display_model_comparison(model_results, list_columns, numeric_columns)
    
    print("\n\n===== DEMOGRAPHIC ANALYSIS =====")
    
    # Calculate demographic statistics for selected models
    demographic_results, included_models = calculate_demographic_stats(
        csv_files, list_columns, numeric_columns, selected_models=selected_models
    )
    
    # Show which models are included in the analysis
    print(f"Analysis includes the following models: {', '.join(included_models)}")
    
    # Display results in tables
    display_demographic_tables(demographic_results, included_models, list_columns, numeric_columns)
    
    # Reset pandas options
    pd.options.mode.chained_assignment = 'warn'  # Reset to default

## Density Forecasts

In [ ]:
import os
import pandas as pd
import glob
import numpy as np
import ast
import re
import json

# Variables to analyze - only the list columns
list_columns = ['answer.Q1_S_Before_list', 'answer.Q2_L_Before_list']

# Model name mapping
model_name_mapping = {
    "DeepSeek-V3(temp=1)": "DeepSeek-V3",
    "Meta-Llama-3-70B-Instruct(temp=0.5)": "Llama3-70B",
    "claude-3-7-sonnet-20250219(temp=0.5)": "Sonnet-3.7",
    "gpt-4o(temp=0.5)": "GPT-4o(t=0.5)",
    "gpt-4o(temp=1)": "GPT-4o(t=1.0)",
    "gpt-4o(temp=1.5)": "GPT-4o(t=1.5)",
    "gpt-4o-mini(temp=1)": "GPT-4o-mini",
    "claude-3-5-haiku-20241022(temp=0.5)": "Haiku-3.5",
    "gpt-4.1(temp=1)": "GPT-4.1",
}

# Function to get simplified model name from filename
def get_model_name(filename):
    for key in model_name_mapping.keys():
        if key in filename:
            return model_name_mapping[key]
    return filename  # Return original if no match found

# Define bin descriptions
bin_descriptions = [
    "Inflation of 12% or more",
    "Inflation between 8% and 12%",
    "Inflation between 4% and 8%",
    "Inflation between 2% and 4%",
    "Inflation between 0% and 2%",
    "Deflation between 0% and 2%",
    "Deflation between 2% and 4%",
    "Deflation between 4% and 8%",
    "Deflation between 8% and 12%",
    "Deflation of 12% or more"
]

# Improved function to safely parse list strings from CSV with better error handling
def parse_list(list_str, model_name=None, row_id=None):
    """
    Parse a string representation of a list into a list of floats.
    Includes special handling for DeepSeek-V3 data with problematic observations.
    """
    if pd.isna(list_str):
        return None
    
    # Skip the problematic observation only for DeepSeek-V3
    if model_name == "DeepSeek-V3" and row_id == 70119139:
        return None
    
    try:
        # Remove any leading/trailing whitespace
        if isinstance(list_str, str):
            list_str = list_str.strip()
            
            parsed_list = None
            
            # Try parsing as JSON first
            try:
                result = json.loads(list_str)
                if isinstance(result, list):
                    parsed_list = result
            except (json.JSONDecodeError, ValueError, TypeError):
                # Try ast.literal_eval
                try:
                    result = ast.literal_eval(list_str)
                    if isinstance(result, list):
                        # Handle nested lists by flattening
                        if result and isinstance(result[0], list):
                            result = result[0]
                        parsed_list = result
                except (SyntaxError, ValueError, TypeError):
                    # Try regex as last resort to extract numbers
                    numbers = re.findall(r'\d+(?:\.\d+)?', list_str)
                    if numbers and len(numbers) >= 10:
                        parsed_list = [float(num) for num in numbers[:10]]
            
            # Check if we have a valid list with 10 elements
            if parsed_list and len(parsed_list) == 10:
                # Convert all elements to float
                numeric_list = []
                
                # For DeepSeek-V3, check for extreme values
                MAX_ALLOWED_VALUE = 100.0  # Set a reasonable maximum value for probabilities
                has_extreme_value = False
                
                for value in parsed_list:
                    try:
                        num_value = float(value)
                        # Only check for extreme values in DeepSeek-V3 data
                        if model_name == "DeepSeek-V3" and num_value > MAX_ALLOWED_VALUE:
                            has_extreme_value = True
                            break
                        numeric_list.append(num_value)
                    except (ValueError, TypeError):
                        # Skip lists with non-numeric elements
                        return None
                
                if model_name == "DeepSeek-V3" and has_extreme_value:
                    return None
                
                if len(numeric_list) == 10:
                    return numeric_list
        
        return None
    except Exception as e:
        print(f"Error parsing list: {e}")
        return None

# Dictionary to store bin averages by model and column
model_bin_averages = {}

# Process each CSV file and collect bin averages
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    short_filename = os.path.splitext(filename)[0]  # Remove extension
    model_name = get_model_name(short_filename)
    
    # Load the CSV file
    df = pd.read_csv(csv_file)
    
    print(f"Processing {model_name}...")
    
    # Initialize model data if not already done
    if model_name not in model_bin_averages:
        model_bin_averages[model_name] = {}
    
    # Process each list column
    for column in list_columns:
        if column not in df.columns:
            print(f"  Column {column} not found in this file")
            continue
        
        # Initialize bin sums and counts
        bin_sums = [0.0] * 10
        bin_counts = [0] * 10
        valid_lists = 0
        
        # Process each row
        for idx, row in df.iterrows():
            if pd.isna(row[column]):
                continue
            
            # Pass the row ID to the parser if available
            row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
            parsed_list = parse_list(row[column], model_name=model_name, row_id=row_id)
            
            if parsed_list and len(parsed_list) == 10:
                valid_lists += 1
                for i, value in enumerate(parsed_list):
                    try:
                        val = float(value)
                        bin_sums[i] += val
                        bin_counts[i] += 1
                    except (ValueError, TypeError):
                        continue
        
        # Calculate averages
        bin_averages = []
        for i in range(10):
            if bin_counts[i] > 0:
                avg = bin_sums[i] / valid_lists  # Use valid_lists as denominator
                bin_averages.append(avg)
            else:
                bin_averages.append(None)
        
        # Store results
        model_bin_averages[model_name][column] = {
            'bin_averages': bin_averages,
            'valid_lists': valid_lists
        }

# Display results
for column in list_columns:
    print(f"\n\n===== Bin Averages for {column} =====")
    
    # Create a table with bins as rows and models as columns
    table_data = []
    all_model_names = sorted(model_bin_averages.keys())
    
    # Headers
    headers = ["Bin"] + all_model_names
    
    # Add rows for each bin
    for i, bin_desc in enumerate(bin_descriptions):
        row = [bin_desc]
        for model_name in all_model_names:
            if model_name in model_bin_averages and column in model_bin_averages[model_name]:
                bin_data = model_bin_averages[model_name][column]
                if bin_data['valid_lists'] > 0 and bin_data['bin_averages'][i] is not None:
                    row.append(f"{bin_data['bin_averages'][i]:.2f}")
                else:
                    row.append("N/A")
            else:
                row.append("N/A")
        table_data.append(row)
    
    # Add a row for sample size
    sample_row = ["Sample Size"]
    for model_name in all_model_names:
        if model_name in model_bin_averages and column in model_bin_averages[model_name]:
            sample_row.append(str(model_bin_averages[model_name][column]['valid_lists']))
        else:
            sample_row.append("N/A")
    table_data.append(sample_row)
    
    # Print table
    print("\n".join([" | ".join([cell.ljust(25) for cell in row]) for row in [headers] + table_data]))

In [ ]:
import os
import pandas as pd
import glob
import numpy as np
import ast
import re
import json
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------- CONFIGURATION ----------------

data_dir = "/Users/alizarif/Downloads/Macbook Air Ali/IU PhD/3rd year paper/V.3 (R&R JME)/Results/Data"
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

list_columns = ['answer.Q1_S_Before_list', 'answer.Q2_L_Before_list']

model_name_mapping = {
    "DeepSeek-V3(temp=1)": "DeepSeek-V3",
    "Meta-Llama-3-70B-Instruct(temp=0.5)": "Llama3-70B",
    "claude-3-7-sonnet-20250219(temp=0.5)": "Sonnet-3.7",
    "gpt-4o(temp=0.5)": "GPT-4o(t=0.5)",
    "gpt-4o(temp=1)": "GPT-4o(t=1.0)",
    "gpt-4o(temp=1.5)": "GPT-4o(t=1.5)",
    "gpt-4o-mini(temp=1)": "GPT-4o-mini",
    "claude-3-5-haiku-20241022(temp=0.5)": "Haiku-3.5",
    "gpt-4.1(temp=1)": "GPT-4.1",
}

bin_short_labels = [
    "Inf. ≥12%", "Inf. 8-12%", "Inf. 4-8%", "Inf. 2-4%", "Inf. 0-2%",
    "Def. 0-2%", "Def. 2-4%", "Def. 4-8%", "Def. 8-12%", "Def. ≥12%"
]

# ---------------- DATA PROCESSING ----------------

def get_model_name(filename):
    for key in model_name_mapping:
        if key in filename:
            return model_name_mapping[key]
    return filename

def parse_list(list_str, model_name=None, row_id=None):
    if pd.isna(list_str):
        return None
    if model_name == "DeepSeek-V3" and row_id == 70119139:
        return None
    try:
        list_str = str(list_str).strip()
        parsed_list = None
        try:
            result = json.loads(list_str)
            if isinstance(result, list):
                parsed_list = result
        except:
            try:
                result = ast.literal_eval(list_str)
                if isinstance(result, list):
                    parsed_list = result[0] if isinstance(result[0], list) else result
            except:
                numbers = re.findall(r'\d+(?:\.\d+)?', list_str)
                if numbers and len(numbers) >= 10:
                    parsed_list = [float(num) for num in numbers[:10]]
        if parsed_list and len(parsed_list) == 10:
            MAX_ALLOWED_VALUE = 100.0
            numeric_list = []
            for value in parsed_list:
                num = float(value)
                if model_name == "DeepSeek-V3" and num > MAX_ALLOWED_VALUE:
                    return None
                numeric_list.append(num)
            return numeric_list if len(numeric_list) == 10 else None
    except:
        return None

def build_model_bin_averages(csv_files):
    model_bin_averages = {}
    for csv_file in csv_files:
        filename = os.path.basename(csv_file)
        model_name = get_model_name(filename)

        df = pd.read_csv(csv_file)
        if model_name not in model_bin_averages:
            model_bin_averages[model_name] = {}

        for column in list_columns:
            if column not in df.columns:
                continue

            bin_sums = [0.0] * 10
            bin_counts = [0] * 10
            valid_lists = 0

            for idx, row in df.iterrows():
                row_id = row.get('agent.userid', None)
                parsed = parse_list(row[column], model_name=model_name, row_id=row_id)
                if parsed:
                    valid_lists += 1
                    for i, val in enumerate(parsed):
                        bin_sums[i] += float(val)
                        bin_counts[i] += 1

            bin_averages = [
                (bin_sums[i] / valid_lists if valid_lists > 0 else None)
                for i in range(10)
            ]
            model_bin_averages[model_name][column] = {
                'bin_averages': bin_averages,
                'valid_lists': valid_lists
            }
    return model_bin_averages

# ---------------- PLOTTING ----------------

def build_single_column_heat_data(model_bin_averages, models_to_include, column):
    heat_data = []
    model_labels = []

    for model in models_to_include:
        model_data = model_bin_averages.get(model, {})
        if column in model_data and model_data[column]['valid_lists'] > 0:
            row = model_data[column]['bin_averages']
            heat_data.append(row)
            total_samples = model_data[column]['valid_lists']
            model_labels.append(f"{model} (n≈{total_samples})")
    return np.array(heat_data), model_labels

def plot_compact_heatmap(model_bin_averages, models_to_include, column, title):
    matrix, labels = build_single_column_heat_data(model_bin_averages, models_to_include, column)
    if matrix.size == 0:
        print(f"No data for {title}")
        return

    plt.figure(figsize=(10, len(labels) * 0.4 + 1))
    sns.heatmap(matrix, annot=True, fmt=".1f", cmap="YlGnBu",
                xticklabels=bin_short_labels, yticklabels=labels,
                cbar_kws={'label': 'Avg. Probability (%)'})
    plt.title(title, fontsize=14)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=10)
    plt.xlabel("Inflation/Deflation Bins")
    plt.ylabel("Model")
    plt.tight_layout()
    plt.show()

# ---------------- RUN ----------------

print("Processing all models...")
model_bin_averages = build_model_bin_averages(csv_files)

selected_models = ["GPT-4.1", "GPT-4o(t=1.0)", "Sonnet-3.7", "Llama3-70B", "DeepSeek-V3", "GPT-4o-mini", "Haiku-3.5"]
gpt4o_models = ["GPT-4o(t=0.5)", "GPT-4o(t=1.0)", "GPT-4o(t=1.5)"]

# Selected models
plot_compact_heatmap(model_bin_averages, selected_models, 'answer.Q1_S_Before_list', "Short-Run Expectations")
plot_compact_heatmap(model_bin_averages, selected_models, 'answer.Q2_L_Before_list', "Long-Run Expectations")

# GPT-4o variants
plot_compact_heatmap(model_bin_averages, gpt4o_models, 'answer.Q1_S_Before_list', "Short-Run Expectations: GPT-4o Temperatures")
plot_compact_heatmap(model_bin_averages, gpt4o_models, 'answer.Q2_L_Before_list', "Long-Run Expectations: GPT-4o Temperatures")


## Mean vs Standard Deviation Scatterplot
#### Figure 2: Standard Deviation vs. Mean of Expectations Before and After Treatment

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Focus only on selected models with refined colors
model_colors = {
    "DeepSeek-V3": '#A9A9A9',    # Darker gray
    "Llama3-70B": '#FF3333',     # Brighter red
    "Sonnet-3.7": '#555555',     # Very dark gray
    #"GPT-4o": '#3333FF',  # Brighter blue 
    "GPT-4.1": '#FF9900'         # Orange-gold
}

# Create focused plots with improved transparency
def create_focused_plots(results_by_variable, total_results):
    # Define the variables for each plot
    short_run_vars = ['answer.Q1_S_Before_list', 'answer.Q1_S_After']
    long_run_vars = ['answer.Q2_L_Before_list', 'answer.Q2_L_After']
    
    # Create a figure with 1x2 layout
    fig, axs = plt.subplots(1, 2, figsize=(16, 7), facecolor='white')
    
    # Customize figure appearance
    plt.rcParams['font.family'] = 'Arial'
    
    # Function to apply threshold filtering
    def filter_extreme_values(variable, std_dev):
        threshold = 5.0
        return None if std_dev > threshold else std_dev
    
    # Process short run plot (left)
    ax = axs[0]
    ax.set_facecolor('#f5f5f5')  # Light gray background
    plot_data(ax, results_by_variable, short_run_vars, filter_extreme_values)
    ax.set_title('Short Run Expectations Before vs After Treatment', fontsize=14, fontweight='bold')
    
    # Process long run plot (right)
    ax = axs[1]
    ax.set_facecolor('#f5f5f5')  # Light gray background
    plot_data(ax, results_by_variable, long_run_vars, filter_extreme_values)
    ax.set_title('Long Run Expectations Before vs After Treatment', fontsize=14, fontweight='bold')
    
    # Create legend
    handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color, 
                     markeredgecolor='black', markersize=10, label=model) 
              for model, color in model_colors.items()]
    
    # Add marker info to legend
    handles.append(Line2D([0], [0], marker='o', color='black', markerfacecolor='none', 
                         markersize=10, label='Before'))
    handles.append(Line2D([0], [0], marker='s', color='black', markerfacecolor='none', 
                         markersize=10, label='After'))
    
    # Add legend to the right of the figure
    fig.legend(handles=handles, loc='center right', bbox_to_anchor=(1, 0.5),
               frameon=True, title="Models", fontsize=11, 
               title_fontsize=13, edgecolor='gray')
    
    # Set overall title
    fig.suptitle('', fontsize=16, fontweight='bold')
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    plt.savefig('focused_transparent_plots.png', dpi=300, bbox_inches='tight')
    plt.show()

# Helper function to plot data for a set of variables
def plot_data(ax, results_by_variable, variables, filter_func):
    # Selected models only
    selected_models = list(model_colors.keys())
    
    # Track data points for markers
    before_points = []
    after_points = []
    
    # Loop through treatments and models for each variable
    for variable in variables:
        if variable not in results_by_variable:
            continue
            
        is_before = '_Before' in variable
        
        for treatment in sorted(results_by_variable[variable].keys()):
            for model_name in sorted(results_by_variable[variable][treatment].keys()):
                # Skip models not in our selected list
                if model_name not in selected_models:
                    continue
                    
                stats = results_by_variable[variable][treatment][model_name]
                
                if stats['mean'] is not None and stats['std_dev'] is not None:
                    x = stats['mean']
                    y = filter_func(variable, stats['std_dev'])
                    
                    if y is None:
                        continue
                    
                    color = model_colors.get(model_name, '#999999')
                    
                    # Use hollow markers for "Before" and filled markers for "After"
                    if is_before:
                        ax.scatter(x, y, s=100, facecolor=color, edgecolors='black', 
                                  marker='o', alpha=0.9, linewidths=1.2)
                        before_points.append((x, y, color, model_name, treatment))
                    else:
                        ax.scatter(x, y, s=100, facecolor=color, edgecolors='black', 
                                  marker='s', alpha=0.9, linewidths=1.2)
                        after_points.append((x, y, color, model_name, treatment))
    
    # Draw lines connecting before and after points for the same model and treatment
    for b_point in before_points:
        for a_point in after_points:
            # Connect points for same model AND same treatment
            if b_point[3] == a_point[3] and b_point[4] == a_point[4]:
                ax.plot([b_point[0], a_point[0]], [b_point[1], a_point[1]], 
                       color=b_point[2], linestyle='-', alpha=0.15, linewidth=2)
    
    # Configure axes
    ax.set_xlabel('Mean', fontsize=12)
    ax.set_ylabel('Standard Deviation', fontsize=12)
    ax.grid(True, linestyle='-', color='#DDDDDD', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Execute the function 
create_focused_plots(results_by_variable, total_results)

## Average Treatment Effects
#### Table 3 :Changes in Inflation Expectations and Treatment Effects Across LLMs

In [ ]:
import os
import pandas as pd
import numpy as np
from tabulate import tabulate
import glob
import re

# Variables to analyze
short_before = 'answer.Q1_S_Before_list'
short_after = 'answer.Q1_S_After'
long_before = 'answer.Q2_L_Before_list'
long_after = 'answer.Q2_L_After'

# Model name mapping
model_name_mapping = {
    "DeepSeek-V3(temp=1)": "DeepSeek-V3",
    "Meta-Llama-3-70B-Instruct(temp=0.5)": "Llama3-70B",
    "claude-3-7-sonnet-20250219(temp=0.5)": "Sonnet-3.7",
    "gpt-4o(temp=0.5)": "GPT-4o(t=0.5)",
    "gpt-4o(temp=1)": "GPT-4o(t=1.0)",
    "gpt-4o(temp=1.5)": "GPT-4o(t=1.5)",
    "gpt-4o-mini(temp=1)": "GPT-4o-mini",
    "claude-3-5-haiku-20241022(temp=0.5)": "Haiku-3.5",
    "gpt-4.1(temp=1)": "GPT-4.1",
}

# Selected models for analysis (excluding GPT-4o with temps 0.5 and 1.5)
selected_models = [
    "DeepSeek-V3", "GPT-4.1", "GPT-4o(t=1.0)", "GPT-4o-mini", 
    "Haiku-3.5", "Llama3-70B", "Sonnet-3.7", "GPT-4o(t=0.5)"
    "GPT-4o(t=1.5)"  # Excluded from analysis
]

# Function to get simplified model name from filename
def get_model_name(filename):
    for key in model_name_mapping.keys():
        if key in filename:
            return model_name_mapping[key]
    return filename  # Return original if no match found

# Function for safe parsing of list strings
def parse_list(list_str, row_id=None):
    """
    Parse a string representation of a list into a list of floats.
    Skips known problematic observations.
    """
    if pd.isna(list_str):
        return None
    
    # Skip the specific problematic observation 
    if row_id == 70119139:
        return None
    
    try:
        # Remove any leading/trailing whitespace
        if isinstance(list_str, str):
            list_str = list_str.strip()
            
            # Try parsing as JSON first
            try:
                import json
                result = json.loads(list_str)
                if isinstance(result, list):
                    # Check for extreme values
                    if any(isinstance(x, (int, float)) and abs(x) > 100000 for x in result):
                        return None
                    # Convert to floats
                    return [float(x) for x in result if isinstance(x, (int, float))]
            except:
                pass
            
            # Try ast.literal_eval
            try:
                import ast
                result = ast.literal_eval(list_str)
                if isinstance(result, list):
                    # Handle nested lists by flattening
                    if result and isinstance(result[0], list):
                        result = result[0]
                    # Check for extreme values
                    if any(isinstance(x, (int, float)) and abs(x) > 100000 for x in result):
                        return None
                    # Convert to floats
                    return [float(x) for x in result if isinstance(x, (int, float))]
            except:
                pass
                
            # Try regex as last resort
            numbers = re.findall(r'\d+(?:\.\d+)?', list_str)
            if numbers and len(numbers) == 10:
                return [float(num) for num in numbers]
        
        return None
    except Exception as e:
        return None

# Function to calculate weighted mean from a list of bin values
def calculate_weighted_mean(parsed_list):
    """Calculate weighted mean using the bin midpoints."""
    if not parsed_list or len(parsed_list) != 10:
        return None
        
    # Define bin midpoints
    bin_midpoints = [
        14.0,  # "Inflation of 12% or more"
        10.0,  # "Inflation between 8% and 12%"
        6.0,   # "Inflation between 4% and 8%"
        3.0,   # "Inflation between 2% and 4%"
        1.0,   # "Inflation between 0% and 2%"
        -1.0,  # "Deflation between 0% and 2%"
        -3.0,  # "Deflation between 2% and 4%"
        -6.0,  # "Deflation between 4% and 8%"
        -10.0, # "Deflation between 8% and 12%"
        -14.0  # "Deflation of 12% or more"
    ]
    
    # Calculate weighted mean
    total_weight = sum(parsed_list)
    if total_weight == 0:
        return None
    
    weighted_sum = sum(val * bin_midpoints[i] for i, val in enumerate(parsed_list))
    return weighted_sum / total_weight

# Dictionary to store results 
results = {}
treatment_effects = {}

# Process each CSV file
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    short_filename = os.path.splitext(filename)[0]  # Remove extension
    model_name = get_model_name(short_filename)
    
    # Skip if not in selected models
    if model_name not in selected_models:
        continue
    
    print(f"\nProcessing {filename}...")
    
    # Load the CSV file
    df = pd.read_csv(csv_file)
    
    # Initialize results for this model if not exists
    if model_name not in results:
        results[model_name] = {}
        treatment_effects[model_name] = {}
    
    # Group by treatment
    if 'scenario.treatment' in df.columns:
        treatments = sorted(df['scenario.treatment'].dropna().unique())
        
        for treatment in treatments:
            treatment_df = df[df['scenario.treatment'] == treatment]
            
            # Initialize treatment results if not exists
            if treatment not in results[model_name]:
                results[model_name][treatment] = {
                    'short_before': None,
                    'short_after': None,
                    'long_before': None,
                    'long_after': None,
                    'short_change': None,
                    'long_change': None,
                    'n_short': 0,
                    'n_long': 0
                }
            
            # Calculate SHORT-TERM before-after differences
            if short_before in df.columns and short_after in df.columns:
                # Process "before" values (distribution forecasts)
                before_values = []
                for idx, row in treatment_df.iterrows():
                    if pd.isna(row[short_before]):
                        continue
                    
                    # Get row ID for checking problematic observations
                    row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                    
                    # Parse list and calculate weighted mean
                    parsed_list = parse_list(row[short_before], row_id)
                    if parsed_list and len(parsed_list) == 10:
                        mean = calculate_weighted_mean(parsed_list)
                        if mean is not None:
                            before_values.append(mean)
                
                # Process "after" values (point estimates)
                after_values = []
                for idx, row in treatment_df.iterrows():
                    if pd.isna(row[short_after]):
                        continue
                    
                    # Check for problematic observations
                    row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                    if row_id == 70119139:
                        continue
                    
                    try:
                        val = float(row[short_after])
                        if abs(val) < 100000:  # Skip extreme values
                            after_values.append(val)
                    except:
                        continue
                
                # Calculate averages
                if before_values and after_values:
                    before_mean = np.mean(before_values)
                    after_mean = np.mean(after_values)
                    change = after_mean - before_mean
                    
                    results[model_name][treatment]['short_before'] = before_mean
                    results[model_name][treatment]['short_after'] = after_mean
                    results[model_name][treatment]['short_change'] = change
                    results[model_name][treatment]['n_short'] = min(len(before_values), len(after_values))
            
            # Calculate LONG-TERM before-after differences
            if long_before in df.columns and long_after in df.columns:
                # Process "before" values (distribution forecasts)
                before_values = []
                for idx, row in treatment_df.iterrows():
                    if pd.isna(row[long_before]):
                        continue
                    
                    # Get row ID for checking problematic observations
                    row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                    
                    # Parse list and calculate weighted mean
                    parsed_list = parse_list(row[long_before], row_id)
                    if parsed_list and len(parsed_list) == 10:
                        mean = calculate_weighted_mean(parsed_list)
                        if mean is not None:
                            before_values.append(mean)
                
                # Process "after" values (point estimates)
                after_values = []
                for idx, row in treatment_df.iterrows():
                    if pd.isna(row[long_after]):
                        continue
                    
                    # Check for problematic observations
                    row_id = row['agent.userid'] if 'agent.userid' in df.columns else None
                    if row_id == 70119139:
                        continue
                    
                    try:
                        val = float(row[long_after])
                        if abs(val) < 100000:  # Skip extreme values
                            after_values.append(val)
                    except:
                        continue
                
                # Calculate averages
                if before_values and after_values:
                    before_mean = np.mean(before_values)
                    after_mean = np.mean(after_values)
                    change = after_mean - before_mean
                    
                    results[model_name][treatment]['long_before'] = before_mean
                    results[model_name][treatment]['long_after'] = after_mean
                    results[model_name][treatment]['long_change'] = change
                    results[model_name][treatment]['n_long'] = min(len(before_values), len(after_values))

# Calculate Average Treatment Effects (ATE) relative to T_0 (control group)
for model_name in results:
    # Skip if no T_0 (control) group
    if 'T_0' not in results[model_name]:
        continue
    
    # Get baseline changes from control group (T_0)
    control_short_change = results[model_name]['T_0']['short_change']
    control_long_change = results[model_name]['T_0']['long_change']
    
    # Calculate ATEs for all other treatments
    for treatment in results[model_name]:
        if treatment == 'T_0':
            # Control group ATE is 0 by definition
            treatment_effects[model_name][treatment] = {
                'short_ate': 0.0,
                'long_ate': 0.0
            }
        else:
            # Calculate ATE as difference from control group
            short_change = results[model_name][treatment]['short_change']
            long_change = results[model_name][treatment]['long_change']
            
            # Calculate ATEs
            short_ate = short_change - control_short_change if short_change is not None and control_short_change is not None else None
            long_ate = long_change - control_long_change if long_change is not None and control_long_change is not None else None
            
            treatment_effects[model_name][treatment] = {
                'short_ate': short_ate,
                'long_ate': long_ate
            }

# Display results: Before-After Changes by Treatment and Model
print("\n===== Changes in Inflation Expectations (After - Before) =====")
for model_name in sorted(results.keys()):
    print(f"\nModel: {model_name}")
    
    # Create table data
    table_data = []
    for treatment in sorted(results[model_name].keys()):
        t_results = results[model_name][treatment]
        row = [
            treatment,
            f"{t_results['short_before']:.2f}" if t_results['short_before'] is not None else "N/A",
            f"{t_results['short_after']:.2f}" if t_results['short_after'] is not None else "N/A",
            f"{t_results['short_change']:.2f}" if t_results['short_change'] is not None else "N/A",
            f"{t_results['n_short']}" if t_results['n_short'] > 0 else "N/A",
            f"{t_results['long_before']:.2f}" if t_results['long_before'] is not None else "N/A",
            f"{t_results['long_after']:.2f}" if t_results['long_after'] is not None else "N/A",
            f"{t_results['long_change']:.2f}" if t_results['long_change'] is not None else "N/A",
            f"{t_results['n_long']}" if t_results['n_long'] > 0 else "N/A"
        ]
        table_data.append(row)
    
    # Display table
    headers = ["Treatment", "Short Before", "Short After", "Short Change", "N Short", 
               "Long Before", "Long After", "Long Change", "N Long"]
    print(tabulate(table_data, headers=headers, tablefmt='grid'))

# Display results: Average Treatment Effects (ATE) by Treatment and Model
print("\n===== Average Treatment Effects (relative to T_0) =====")
for model_name in sorted(treatment_effects.keys()):
    print(f"\nModel: {model_name}")
    
    # Create table data
    table_data = []
    for treatment in sorted(treatment_effects[model_name].keys()):
        t_effects = treatment_effects[model_name][treatment]
        row = [
            treatment,
            f"{t_effects['short_ate']:.2f}" if t_effects['short_ate'] is not None else "N/A",
            f"{t_effects['long_ate']:.2f}" if t_effects['long_ate'] is not None else "N/A"
        ]
        table_data.append(row)
    
    # Display table
    headers = ["Treatment", "Short-Term ATE", "Long-Term ATE"]
    print(tabulate(table_data, headers=headers, tablefmt='grid'))

# Create Latex Tables for ATEs
print("\n\n===== LaTeX Table for Average Treatment Effects =====")

# Function to create LaTeX table for ATEs
def create_ate_latex_table():
    latex_table = []
    latex_table.append("\\begin{table}[htbp]")
    latex_table.append("\\centering")
    latex_table.append("\\caption{Average Treatment Effects on Inflation Expectations}")
    latex_table.append("\\label{tab:treatment_effects}")
    latex_table.append("\\small")
    
    # Create table header with all selected models
    header_line = "\\begin{tabular}{l"
    for _ in range(len(selected_models) * 2):  # 2 columns per model (short and long)
        header_line += "c"
    header_line += "}"
    latex_table.append(header_line)
    
    latex_table.append("\\toprule")
    
    # Create model headers spanning 2 columns each
    model_header = "\\multirow{2}{*}{Treatment}"
    for model in selected_models:
        model_header += f" & \\multicolumn{{2}}{{c}}{{{model}}}"
    latex_table.append(model_header + " \\\\")
    
    # Create Short/Long term headers
    term_header = ""
    for _ in range(len(selected_models)):
        term_header += " & Short & Long"
    latex_table.append(term_header + " \\\\")
    
    latex_table.append("\\midrule")
    
    # Add rows for each treatment
    all_treatments = set()
    for model in treatment_effects:
        all_treatments.update(treatment_effects[model].keys())
    
    for treatment in sorted(all_treatments):
        row = f"{treatment}"
        for model in selected_models:
            if model in treatment_effects and treatment in treatment_effects[model]:
                t_effects = treatment_effects[model][treatment]
                short_ate = f"{t_effects['short_ate']:.2f}" if t_effects['short_ate'] is not None else "N/A"
                long_ate = f"{t_effects['long_ate']:.2f}" if t_effects['long_ate'] is not None else "N/A"
                row += f" & {short_ate} & {long_ate}"
            else:
                row += " & N/A & N/A"
        latex_table.append(row + " \\\\")
    
    latex_table.append("\\bottomrule")
    latex_table.append("\\end{tabular}")
    
    # Add notes
    latex_table.append("\\begin{tablenotes}")
    latex_table.append("\\small")
    latex_table.append("\\item \\textit{Notes:} This table reports Average Treatment Effects (ATEs) for each treatment relative to the control group (T\\_0). Short refers to short-term (1-year) inflation expectations and Long refers to long-term (2-year) inflation expectations. ATEs are calculated as the difference between the treatment group's change in expectations (after - before) and the control group's change in expectations.")
    latex_table.append("\\end{tablenotes}")
    latex_table.append("\\end{table}")
    
    return "\n".join(latex_table)

# Print LaTeX table
print(create_ate_latex_table())

# Create summary tables for publication
print("\n\n===== LaTeX Summary Table for Publication =====")

# Function to create a simplified summary table for publication
def create_summary_latex_table():
    latex_table = []
    latex_table.append("\\begin{table}[htbp]")
    latex_table.append("\\centering")
    latex_table.append("\\caption{Treatment Effects on Inflation Expectations Across LLMs}")
    latex_table.append("\\label{tab:treatment_effects_summary}")
    latex_table.append("\\small")
    
    # Create table with 3 groups of models (OpenAI, Anthropic, Open Source)
    latex_table.append("\\begin{tabular}{lcccccc}")
    latex_table.append("\\toprule")
    
    # Create headers with model groupings
    latex_table.append("\\multirow{2}{*}{Treatment} & \\multicolumn{2}{c}{OpenAI Models} & \\multicolumn{2}{c}{Anthropic Models} & \\multicolumn{2}{c}{Open Source Models} \\\\")
    latex_table.append("\\cmidrule(lr){2-3} \\cmidrule(lr){4-5} \\cmidrule(lr){6-7}")
    latex_table.append("& Short-Term & Long-Term & Short-Term & Long-Term & Short-Term & Long-Term \\\\")
    latex_table.append("\\midrule")
    
    # Model groupings
    openai_models = ["GPT-4.1", "GPT-4o(t=1.0)", "GPT-4o-mini"]
    anthropic_models = ["Haiku-3.5", "Sonnet-3.7"]
    opensource_models = ["DeepSeek-V3", "Llama3-70B"]
    
    # Add rows for each treatment
    all_treatments = set()
    for model in treatment_effects:
        all_treatments.update(treatment_effects[model].keys())
    
    for treatment in sorted(all_treatments):
        # Skip T_0 since ATEs are 0 by definition
        if treatment == "T_0":
            continue
            
        row = f"{treatment}"
        
        # OpenAI Models (average)
        openai_short = []
        openai_long = []
        for model in openai_models:
            if model in treatment_effects and treatment in treatment_effects[model]:
                t_effects = treatment_effects[model][treatment]
                if t_effects['short_ate'] is not None:
                    openai_short.append(t_effects['short_ate'])
                if t_effects['long_ate'] is not None:
                    openai_long.append(t_effects['long_ate'])
        
        openai_short_avg = np.mean(openai_short) if openai_short else None
        openai_long_avg = np.mean(openai_long) if openai_long else None
        
        row += f" & {openai_short_avg:.2f}" if openai_short_avg is not None else " & N/A"
        row += f" & {openai_long_avg:.2f}" if openai_long_avg is not None else " & N/A"
        
        # Anthropic Models (average)
        anthropic_short = []
        anthropic_long = []
        for model in anthropic_models:
            if model in treatment_effects and treatment in treatment_effects[model]:
                t_effects = treatment_effects[model][treatment]
                if t_effects['short_ate'] is not None:
                    anthropic_short.append(t_effects['short_ate'])
                if t_effects['long_ate'] is not None:
                    anthropic_long.append(t_effects['long_ate'])
        
        anthropic_short_avg = np.mean(anthropic_short) if anthropic_short else None
        anthropic_long_avg = np.mean(anthropic_long) if anthropic_long else None
        
        row += f" & {anthropic_short_avg:.2f}" if anthropic_short_avg is not None else " & N/A"
        row += f" & {anthropic_long_avg:.2f}" if anthropic_long_avg is not None else " & N/A"
        
        # Open Source Models (average)
        os_short = []
        os_long = []
        for model in opensource_models:
            if model in treatment_effects and treatment in treatment_effects[model]:
                t_effects = treatment_effects[model][treatment]
                if t_effects['short_ate'] is not None:
                    os_short.append(t_effects['short_ate'])
                if t_effects['long_ate'] is not None:
                    os_long.append(t_effects['long_ate'])
        
        os_short_avg = np.mean(os_short) if os_short else None
        os_long_avg = np.mean(os_long) if os_long else None
        
        row += f" & {os_short_avg:.2f}" if os_short_avg is not None else " & N/A"
        row += f" & {os_long_avg:.2f}" if os_long_avg is not None else " & N/A"
        
        latex_table.append(row + " \\\\")
    
    latex_table.append("\\bottomrule")
    latex_table.append("\\end{tabular}")
    
    # Add notes
    latex_table.append("\\begin{tablenotes}")
    latex_table.append("\\small")
    latex_table.append("\\item \\textit{Notes:} This table reports average Treatment Effects (TEs) by model family, relative to the control group (T\\_0). Short-Term refers to 1-year inflation expectations and Long-Term to 2-year expectations. OpenAI models include GPT-4.1, GPT-4o, and GPT-4o-mini; Anthropic models include Claude 3.5 Haiku and Claude 3.7 Sonnet; Open Source models include DeepSeek-V3 and Llama3-70B.")
    latex_table.append("\\end{tablenotes}")
    latex_table.append("\\end{table}")
    
    return "\n".join(latex_table)

# Print summary LaTeX table
print(create_summary_latex_table())

## Empirical Model
#### Table 4: Treatment Effects on Short-Run and Long-Run Inflation Expectations
#### Table 5: Demographic Effects on Short-Run and Long-Run Inflation Expectations

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import ast

# Bin midpoints for weighted mean
bin_midpoints = [14.0, 10.0, 6.0, 3.0, 1.0, -1.0, -3.0, -6.0, -10.0, -14.0]

# Weighted mean calculator
def calculate_weighted_mean(bin_values):
    try:
        if isinstance(bin_values, str):
            bin_values = ast.literal_eval(bin_values)
        if len(bin_values) != len(bin_midpoints):
            return np.nan
        total_weight = sum(bin_values)
        if total_weight == 0:
            return np.nan
        weighted_sum = sum(v * m for v, m in zip(bin_values, bin_midpoints))
        return weighted_sum / total_weight
    except:
        return np.nan

# Function to categorize age
def categorize_age(age):
    if pd.isna(age):
        return np.nan
    elif 18 <= age <= 30:
        return "18-30"
    elif 31 <= age <= 40:
        return "31-40"
    elif 41 <= age <= 50:
        return "41-50"
    elif 51 <= age <= 60:
        return "51-60"
    elif age > 60:
        return "60+"
    else:
        return np.nan

# Function to categorize income with Under 50k as base
def categorize_income(income_category):
    if pd.isna(income_category):
        return np.nan
    elif income_category == "Under 50k":
        return "Under_50k"
    elif income_category == "50k to 100k":
        return "50k_to_100k"
    elif income_category == "Over 100k":
        return "Over_100k"
    else:
        return np.nan

# Regression function
def run_regression(df, pre_col, post_col):
    formula = (
        f"{post_col} ~ {pre_col} + C(treatment) + "
        f"C(treatment):{pre_col} + "
        "C(agent_marital_text) + C(agent_education) + C(agent_gender_text) + "
        "C(age_group, Treatment('18-30')) + C(income_category, Treatment('Under_50k'))"
    )
    model = smf.ols(formula=formula, data=df).fit(cov_type='HC3')
    return model.summary()

# Process each CSV
for file_path in csv_files:
    print(f"\n=== Processing file: {os.path.basename(file_path)} ===\n")
    
    df = pd.read_csv(file_path)
    
    # Rename problematic column names
    df = df.rename(columns=lambda x: x.replace('.', '_'))
    
    # Apply weighted means
    df['short_pre'] = df['answer_Q1_S_Before_list'].apply(calculate_weighted_mean)
    df['long_pre'] = df['answer_Q2_L_Before_list'].apply(calculate_weighted_mean)
    df['short_post'] = df['answer_Q1_S_After']
    df['long_post'] = df['answer_Q2_L_After']
    
    # Categorize age into age groups
    df['age_group'] = df['agent_age'].apply(categorize_age)
    
    # Use existing categorical income values
    df['income_category'] = df['agent_income'].apply(categorize_income)
    
    # Filter valid treatments
    df = df[df['scenario_treatment'].notna()]
    df = df[df['scenario_treatment'] != '']
    
    # Convert to categorical
    df['treatment'] = df['scenario_treatment'].astype('category')
    df['agent_marital_text'] = df['agent_marital_text'].astype('category')
    df['agent_education'] = df['agent_education'].astype('category')
    df['agent_gender_text'] = df['agent_gender_text'].astype('category')
    df['age_group'] = df['age_group'].astype('category')
    df['income_category'] = df['income_category'].astype('category')
    
    # Skip if missing necessary columns
    required_columns = ['short_pre', 'short_post', 'long_pre', 'long_post', 'treatment',
                        'agent_marital_text', 'agent_education', 'agent_gender_text',
                        'age_group', 'income_category']
    if not all(col in df.columns for col in required_columns):
        print("Missing columns. Skipping file.")
        continue
    
    try:
        # Run and display short-run regression
        print("\n--- SHORT-RUN REGRESSION ---\n")
        print(run_regression(df, 'short_pre', 'short_post'))
        
        # Run and display long-run regression
        print("\n--- LONG-RUN REGRESSION ---\n")
        print(run_regression(df, 'long_pre', 'long_post'))
    except Exception as e:
        print(f"Error in regression for {file_path}: {e}")

#### Figure 3: Treatment Effects Across LLMs and Human Survey 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches

def create_coefficient_plot(models_to_include=None):
    """
    Create a compact 2x2 coefficient plot for treatment effects on inflation expectations.
    Right column plots will not have y-axis labels.
    
    Parameters:
    -----------
    models_to_include : list or None
        List of model names to include in the plot. If None, include all models.
        Options: 'Claude 3.7', 'GPT-4.1', 'Llama-3', 'Human Survey'
    """
    if models_to_include is None:
        models_to_include = ['Claude 3.7', 'GPT-4.1', 'Llama-3', 'Human Survey']
    
    # Updated treatment labels for the y-axis based on the image
    treatment_labels = [
        'Intercept',
        'θ - Control Group',
        'Population Growth',
        'Current FFR',
        'FFR + 1Y Proj',
        'FFR + 1Y and Long run',
        '3Y Inflation',
        '1Y Inflation',
        'Infl + 1Y Proj',
        'Infl + 1Y and Long run',
        'Mortgage Rate'
    ]
    
    # Data from the human survey table
    human_intercept = -0.192
    human_intercept_se = 0.024
    human_theta = 0.994
    human_theta_se = 0.003
    
    human_treatments_b = {
        'T1': -0.035, 'T2': 1.316, 'T3': 0.756, 'T4': 2.379, 
        'T5': -0.013, 'T6': 0.020, 'T7': 0.885, 'T8': 0.019, 'T9': 4.465
    }
    
    human_treatments_se_b = {
        'T1': 0.040, 'T2': 0.063, 'T3': 0.050, 'T4': 0.074, 
        'T5': 0.040, 'T6': 0.037, 'T7': 0.054, 'T8': 0.037, 'T9': 0.042
    }
    
    human_treatments_y = {
        'T1': -0.002, 'T2': -0.329, 'T3': -0.176, 'T4': -0.571, 
        'T5': -0.002, 'T6': -0.001, 'T7': -0.216, 'T8': 0.001, 'T9': -0.989
    }
    
    human_treatments_se_y = {
        'T1': 0.006, 'T2': 0.012, 'T3': 0.009, 'T4': 0.015, 
        'T5': 0.006, 'T6': 0.006, 'T7': 0.010, 'T8': 0.005, 'T9': 0.006
    }
    
    # Data for Short-Run (1 year ahead)
    sr_data = {
        'Parameter': treatment_labels,
        'Claude 3.7': [
            1.6534, 0.6522,  # Intercept, Control Group
            0.9405, 0.1526, 0.6615, 0.2962, 0.4293, 0.9954, 1.3417, 0.8587, 0.4730
        ],
        'Claude 3.7_se': [
            0.055, 0.016,  # Standard errors
            0.075, 0.073, 0.074, 0.069, 0.070, 0.072, 0.066, 0.064, 0.067
        ],
        'GPT-4.1': [
            1.0315, 0.5567,  # Intercept, Control Group
            0.3838, -0.1019, 0.1200, 0.2618, 0.2449, 0.5884, 1.1198, 1.3877, 0.1408
        ],
        'GPT-4.1_se': [
            0.040, 0.019,  # Standard errors
            0.051, 0.061, 0.056, 0.056, 0.061, 0.051, 0.051, 0.044, 0.057
        ],
        'Llama-3': [
            3.2749, 0.0678,  # Intercept, Control Group
            -0.6631, -0.2460, -0.0911, 0.1013, 0.5411, -0.8345, -0.4693, -0.6158, 0.7080
        ],
        'Llama-3_se': [
            0.064, 0.024,  # Standard errors
            0.103, 0.112, 0.072, 0.091, 0.080, 0.070, 0.072, 0.074, 0.094
        ],
        # Human Survey data
        'Human Survey': [
            human_intercept, human_theta,  # Intercept, Control Group
            human_treatments_b['T1'], human_treatments_b['T2'], human_treatments_b['T3'], 
            human_treatments_b['T4'], human_treatments_b['T5'], human_treatments_b['T6'], 
            human_treatments_b['T7'], human_treatments_b['T8'], human_treatments_b['T9']
        ],
        'Human Survey_se': [
            human_intercept_se, human_theta_se,  # Standard errors
            human_treatments_se_b['T1'], human_treatments_se_b['T2'], human_treatments_se_b['T3'], 
            human_treatments_se_b['T4'], human_treatments_se_b['T5'], human_treatments_se_b['T6'], 
            human_treatments_se_b['T7'], human_treatments_se_b['T8'], human_treatments_se_b['T9']
        ]
    }
    
    # Data for Long-Run (3 years ahead)
    lr_data = {
        'Parameter': treatment_labels,
        'Claude 3.7': [
            2.3056, 0.2866,  # Intercept, Control Group
            0.1293, -0.2944, -0.1797, -0.0223, 0.0609, 0.0033, -0.2856, -0.1944, -0.0613
        ],
        'Claude 3.7_se': [
            0.034, 0.014,  # Standard errors
            0.042, 0.043, 0.043, 0.038, 0.049, 0.044, 0.039, 0.037, 0.045
        ],
        'GPT-4.1': [
            1.3584, 0.4097,  # Intercept, Control Group
            0.1377, 0.0338, 0.1457, 0.1945, 0.1281, 0.3414, 0.8505, 0.6904, 0.0130
        ],
        'GPT-4.1_se': [
            0.039, 0.017,  # Standard errors
            0.052, 0.051, 0.054, 0.056, 0.054, 0.047, 0.044, 0.039, 0.053
        ],
        'Llama-3': [
            2.8486, 0.0932,  # Intercept, Control Group
            -0.1719, -0.1562, -0.2190, -0.3920, 0.3720, -0.5457, -0.0344, -0.8245, 0.8110
        ],
        'Llama-3_se': [
            0.031, 0.015,  # Standard errors
            0.055, 0.040, 0.035, 0.035, 0.044, 0.037, 0.035, 0.031, 0.045
        ],
        # Human Survey data
        'Human Survey': [
            human_intercept, human_theta,  # Intercept, Control Group
            human_treatments_b['T1'], human_treatments_b['T2'], human_treatments_b['T3'], 
            human_treatments_b['T4'], human_treatments_b['T5'], human_treatments_b['T6'], 
            human_treatments_b['T7'], human_treatments_b['T8'], human_treatments_b['T9']
        ],
        'Human Survey_se': [
            human_intercept_se, human_theta_se,  # Standard errors
            human_treatments_se_b['T1'], human_treatments_se_b['T2'], human_treatments_se_b['T3'], 
            human_treatments_se_b['T4'], human_treatments_se_b['T5'], human_treatments_se_b['T6'], 
            human_treatments_se_b['T7'], human_treatments_se_b['T8'], human_treatments_se_b['T9']
        ]
    }
    
    # Slope coefficients for SR
    sr_slope_data = {
        'Parameter': treatment_labels[2:],  # Skip Intercept and Control Group
        'Claude 3.7': [
            -0.3525, -0.0657, -0.2215, -0.0846, -0.1119, -0.4036, -0.5713, -0.4591, -0.1863
        ],
        'Claude 3.7_se': [
            0.023, 0.022, 0.023, 0.020, 0.021, 0.022, 0.020, 0.019, 0.020
        ],
        'GPT-4.1': [
            -0.1496, 0.0734, -0.0333, -0.0764, 0.0018, -0.1794, -0.3164, -0.4394, -0.0731
        ],
        'GPT-4.1_se': [
            0.025, 0.029, 0.028, 0.028, 0.030, 0.025, 0.024, 0.022, 0.028
        ],
        'Llama-3': [
            -0.0337, 0.0355, -0.0140, -0.1923, -0.1011, -0.0513, -0.0620, -0.0550, 0.0057
        ],
        'Llama-3_se': [
            0.038, 0.042, 0.027, 0.035, 0.030, 0.026, 0.027, 0.027, 0.035
        ],
        'Human Survey': [
            human_treatments_y['T1'], human_treatments_y['T2'], human_treatments_y['T3'], 
            human_treatments_y['T4'], human_treatments_y['T5'], human_treatments_y['T6'], 
            human_treatments_y['T7'], human_treatments_y['T8'], human_treatments_y['T9']
        ],
        'Human Survey_se': [
            human_treatments_se_y['T1'], human_treatments_se_y['T2'], human_treatments_se_y['T3'], 
            human_treatments_se_y['T4'], human_treatments_se_y['T5'], human_treatments_se_y['T6'], 
            human_treatments_se_y['T7'], human_treatments_se_y['T8'], human_treatments_se_y['T9']
        ]
    }
    
    # Slope coefficients for LR
    lr_slope_data = {
        'Parameter': treatment_labels[2:],  # Skip Intercept and Control Group
        'Claude 3.7': [
            -0.0529, 0.0557, 0.0003, -0.0239, 0.0492, -0.0168, 0.0089, -0.2478, 0.0181
        ],
        'Claude 3.7_se': [
            0.017, 0.017, 0.017, 0.015, 0.020, 0.017, 0.016, 0.015, 0.018
        ],
        'GPT-4.1': [
            -0.0716, -0.0059, -0.0461, -0.0412, -0.0415, -0.1054, -0.2620, -0.3778, -0.0333
        ],
        'GPT-4.1_se': [
            0.024, 0.023, 0.024, 0.025, 0.025, 0.022, 0.020, 0.018, 0.024
        ],
        'Llama-3': [
            -0.0506, -0.0517, -0.0666, -0.1065, -0.0406, -0.0789, -0.0988, -0.0936, -0.0669
        ],
        'Llama-3_se': [
            0.029, 0.021, 0.019, 0.018, 0.024, 0.019, 0.018, 0.016, 0.024
        ],
        'Human Survey': [
            human_treatments_y['T1'], human_treatments_y['T2'], human_treatments_y['T3'], 
            human_treatments_y['T4'], human_treatments_y['T5'], human_treatments_y['T6'], 
            human_treatments_y['T7'], human_treatments_y['T8'], human_treatments_y['T9']
        ],
        'Human Survey_se': [
            human_treatments_se_y['T1'], human_treatments_se_y['T2'], human_treatments_se_y['T3'], 
            human_treatments_se_y['T4'], human_treatments_se_y['T5'], human_treatments_se_y['T6'], 
            human_treatments_se_y['T7'], human_treatments_se_y['T8'], human_treatments_se_y['T9']
        ]
    }
    
    # Create DataFrames
    sr_df = pd.DataFrame(sr_data)
    lr_df = pd.DataFrame(lr_data)
    sr_slope_df = pd.DataFrame(sr_slope_data)
    lr_slope_df = pd.DataFrame(lr_slope_data)
    
    # Create figure with 2x2 layout
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('', fontsize=16)
    
    # Titles for the panels
    axs[0, 0].set_title('Short-Run Intercept (beta) - 1 Year Ahead')
    axs[0, 1].set_title('Long-Run Intercept (beta) - 3 Years Ahead')
    axs[1, 0].set_title('Short-Run Slope (gamma) - 1 Year Ahead')
    axs[1, 1].set_title('Long-Run Slope (gamma) - 3 Years Ahead')
    
    # Colors and markers for different models
    colors = {
        'Claude 3.7': 'black',
        'GPT-4.1': 'red',
        'Llama-3': 'blue',
        'Human Survey': 'green'
    }
    
    markers = {
        'Claude 3.7': 'o',
        'GPT-4.1': 's',
        'Llama-3': 'D',
        'Human Survey': '^'
    }
    
    # Add vertical lines at x=0
    for i in range(2):
        for j in range(2):
            axs[i, j].axvline(x=0, color='black', linestyle='--', alpha=0.7)
            axs[i, j].grid(True, linestyle='--', alpha=0.7)
    
    # Plot intercept coefficients
    for model in models_to_include:
        if model in sr_df.columns:
            # Extract data for this model
            sr_coef = sr_df[model].values
            sr_se = sr_df[f'{model}_se'].values
            
            lr_coef = lr_df[model].values
            lr_se = lr_df[f'{model}_se'].values
            
            # Plot short-run intercepts
            for j, param in enumerate(treatment_labels):
                axs[0, 0].errorbar(
                    sr_coef[j], len(treatment_labels) - 1 - j, 
                    xerr=sr_se[j] * 1.96,  # 95% confidence interval
                    fmt=markers[model], 
                    color=colors[model], 
                    markersize=8,
                    capsize=5,
                    label=model if j == 0 else ""
                )
                
                # Long-run intercepts
                axs[0, 1].errorbar(
                    lr_coef[j], len(treatment_labels) - 1 - j, 
                    xerr=lr_se[j] * 1.96,  # 95% confidence interval
                    fmt=markers[model], 
                    color=colors[model], 
                    markersize=8,
                    capsize=5,
                    label=model if j == 0 else ""
                )
    
    # Plot slope coefficients
    for model in models_to_include:
        if model in sr_slope_df.columns:
            # Extract data for this model
            sr_slope = sr_slope_df[model].values
            sr_slope_se = sr_slope_df[f'{model}_se'].values
            
            lr_slope = lr_slope_df[model].values
            lr_slope_se = lr_slope_df[f'{model}_se'].values
            
            # Plot short-run slopes
            for j, param in enumerate(treatment_labels[2:]):
                axs[1, 0].errorbar(
                    sr_slope[j], len(treatment_labels[2:]) - 1 - j, 
                    xerr=sr_slope_se[j] * 1.96,  # 95% confidence interval
                    fmt=markers[model], 
                    color=colors[model], 
                    markersize=8,
                    capsize=5,
                    label=model if j == 0 else ""
                )
                
                # Long-run slopes
                axs[1, 1].errorbar(
                    lr_slope[j], len(treatment_labels[2:]) - 1 - j, 
                    xerr=lr_slope_se[j] * 1.96,  # 95% confidence interval
                    fmt=markers[model], 
                    color=colors[model], 
                    markersize=8,
                    capsize=5,
                    label=model if j == 0 else ""
                )
    
    # Set the y-axis ticks and labels for the LEFT column only
    axs[0, 0].set_yticks(range(len(treatment_labels)-1, -1, -1))
    axs[0, 0].set_yticklabels(treatment_labels)
    
    axs[1, 0].set_yticks(range(len(treatment_labels[2:])-1, -1, -1))
    axs[1, 0].set_yticklabels(treatment_labels[2:])
    
    # For the RIGHT column, set the ticks but hide the labels
    axs[0, 1].set_yticks(range(len(treatment_labels)-1, -1, -1))
    axs[0, 1].set_yticklabels([])  # Hide y-axis labels
    
    axs[1, 1].set_yticks(range(len(treatment_labels[2:])-1, -1, -1))
    axs[1, 1].set_yticklabels([])  # Hide y-axis labels
    
    # Set x-axis labels
    for i in range(2):
        for j in range(2):
            axs[i, j].set_xlabel('Coefficient Value')
    
    # Add a legend at the top in a bordered black box
    handles = [plt.Line2D([0], [0], marker=markers[model], color=colors[model], markersize=8, linestyle='') 
              for model in models_to_include if model in colors]
    legend = fig.legend(handles=handles, labels=models_to_include, 
               loc='upper center', bbox_to_anchor=(0.5, 0.98),
               ncol=len(models_to_include), frameon=True, 
               fancybox=False, edgecolor='black', framealpha=1)
    legend.get_frame().set_linewidth(1)  # Make the border thicker
    
    # Adjust layout and spacing between subplots
    plt.tight_layout(rect=[0, 0.02, 1, 0.92])  # Adjust for top legend
    plt.subplots_adjust(wspace=0.1)  # Reduce horizontal space between plots
    
    return fig

# Create the plot with all models
fig = create_coefficient_plot()
# To create a plot with only specific models:
# fig = create_coefficient_plot(['Claude 3.7', 'Human Survey'])
plt.savefig('treatment_effects_plot_clean.png', dpi=300, bbox_inches='tight')
plt.show()